In [1]:
import os
import psycopg2
from dotenv import load_dotenv

# Ensure system-level or updated .env variables take precedence over existing environment variables
load_dotenv(override=True)

# Database configuration loaded from environment variables for security and flexibility
USER = os.getenv("user")
PASSWORD = os.getenv("password")
HOST = os.getenv("host")
PORT = os.getenv("port")
DBNAME = os.getenv("dbname")

# Initialize connection reference to ensure it is available in the 'finally' safety block
conn = None

try:
    # --- 1. Database Connection Establishment ---
    print(f"Connecting to {HOST}...")
    
    conn = psycopg2.connect(
        user=USER,
        password=PASSWORD,
        host=HOST,
        port=PORT,
        dbname=DBNAME
    )
    
    # Cursor is required to execute queries and manage the context of the database operations
    cur = conn.cursor()

    # --- 2. Schema Definition and Execution ---
    print("Creating 'assets' table...")
    
    # Using 'IF NOT EXISTS' to prevent runtime errors if the script is executed multiple times
    # Financial metrics use DECIMAL(10, 4) to preserve precision and avoid floating-point rounding errors
    cur.execute("""
        CREATE TABLE IF NOT EXISTS public.assets (
            ticker TEXT PRIMARY KEY,
            asset_name TEXT NOT NULL,
            sector TEXT,
            exchange TEXT,
            currency TEXT,
            beta DECIMAL(10, 4),
            dividend_yield DECIMAL(10, 4),
            trailingpe DECIMAL(10, 4),
            pricetobook DECIMAL(10, 4),
            fcf_yield DECIMAL(10, 4),
            revenuegrowth DECIMAL(10, 4),
            earningsgrowth DECIMAL(10, 4),
            forwardeps DECIMAL(10, 4),
            payoutratio DECIMAL(10, 4),
            asset_type TEXT,
            value_score DECIMAL(10, 4),
            growth_score DECIMAL(10, 4),
            dividend_score DECIMAL(10, 4)
        );
    """)

    # --- 3. Transaction Management ---
    # Explicitly commit the DDL transaction to persist the table creation in PostgreSQL
    conn.commit()
    print("Table created successfully.")

except Exception as e:
    # Catch-all block to prevent script crashes and log the specific PostgreSQL/Connection exception
    print(f"Database error: {e}")

finally:
    # --- 4. Resource Cleanup ---
    # Safe teardown of database objects to prevent connection leaks and idle session buildup
    if conn:
        cur.close()
        conn.close()
        print("Connection closed.")

Connecting to aws-1-eu-north-1.pooler.supabase.com...
Creating 'assets' table...
Table created successfully.
Connection closed.
